#### we have cooking actions like:

##### mix
##### stir
##### boil
##### fry
##### bake

while some may appear very frequently, some are rare and it requires a large size of corpus if recipes to include them in the set of unique recipes, hence they form teh heavy tail of the rank frequency distribution. Now our task is we want to test whether their frequency distribution follows a power law lik distribution/ Zipf law:

f(r)∝r power(−β)

where f(r) is the frequency of the ingredient with rank r
this can also be written as

P(r)∝r−α
where P(r) = f(r) / cumulative frequency of all ingredients

In [2]:
# =============================================================================
# IMPROVED ZIPF ANALYSIS
# Distribution Comparison + Bootstrap CI
# =============================================================================

import pandas as pd
import numpy as np
import powerlaw

# =========================================================
# LOAD DATA
# =========================================================

print("Loading data...")

df_actions_raw = pd.read_excel(
    "Datasets/RecipeID_CookingAction_List.xlsx"
)

df_actions_raw = df_actions_raw.dropna(
    subset=['Processes']
)

df_actions_raw['Processes'] = (
    df_actions_raw['Processes']
    .astype(str)
    .str.split(r'\|\|')
)

df_actions = (
    df_actions_raw
    .explode('Processes')
    .reset_index(drop=True)
)

df_actions['Processes'] = (
    df_actions['Processes']
    .str.strip()
)

df_actions = df_actions.rename(
    columns={
        'Recipe_id': 'recipe_no',
        'Processes': 'action_id'
    }
)

# =========================================================
# FREQUENCY DATA
# =========================================================

data = (
    df_actions['action_id']
    .value_counts()
    .values
)

# =========================================================
# FIT POWER LAW
# =========================================================

print("\nFitting power law...")

fit = powerlaw.Fit(
    data,
    discrete=True,
    verbose=False
)

alpha = fit.alpha
sigma = fit.sigma
xmin = fit.xmin

beta = 1.0 / (alpha - 1.0)

# =========================================================
# DISTRIBUTION COMPARISONS
# =========================================================

print("\n--- Distribution Comparisons ---")

comparisons = [
    ('power_law', 'lognormal'),
    ('power_law', 'exponential'),
    ('power_law', 'truncated_power_law')
]

comparison_results = []

for dist1, dist2 in comparisons:

    R, p = fit.distribution_compare(dist1, dist2)

    comparison_results.append({
        'Distribution_1': dist1,
        'Distribution_2': dist2,
        'LogLikelihood_Ratio': R,
        'p_value': p
    })

    print(f"\n{dist1} vs {dist2}")
    print(f"R = {R:.4f}")
    print(f"p = {p:.6f}")

    if p < 0.05:
        if R > 0:
            print(f"→ {dist1} fits significantly better")
        else:
            print(f"→ {dist2} fits significantly better")
    else:
        print("→ No statistically significant difference")

# =========================================================
# BOOTSTRAP CONFIDENCE INTERVAL
# =========================================================

print("\n--- Bootstrap Confidence Interval ---")

n_bootstrap = 200
bootstrap_alphas = []

np.random.seed(42)

for i in range(n_bootstrap):

    # resample with replacement
    sample = np.random.choice(
        data,
        size=len(data),
        replace=True
    )

    try:

        sample_fit = powerlaw.Fit(
            sample,
            discrete=True,
            verbose=False
        )

        bootstrap_alphas.append(
            sample_fit.alpha
        )

    except:
        continue

bootstrap_alphas = np.array(
    bootstrap_alphas
)

ci_lower = np.percentile(
    bootstrap_alphas,
    2.5
)

ci_upper = np.percentile(
    bootstrap_alphas,
    97.5
)

print(f"\nAlpha = {alpha:.6f}")
print(f"Bootstrap 95% CI = [{ci_lower:.6f}, {ci_upper:.6f}]")

# =========================================================
# EXPORT RESULTS
# =========================================================

results_df = pd.DataFrame({
    "xmin": [xmin],
    "alpha": [alpha],
    "sigma": [sigma],
    "beta": [beta],
    "bootstrap_ci_lower": [ci_lower],
    "bootstrap_ci_upper": [ci_upper]
})

results_df.to_csv(
    "improved_zipf_parameters.csv",
    index=False
)

comparison_df = pd.DataFrame(
    comparison_results
)

comparison_df.to_csv(
    "distribution_comparison_results.csv",
    index=False
)

print("\nResults exported successfully.")

Loading data...

Fitting power law...

--- Distribution Comparisons ---

power_law vs lognormal
R = -8.0446
p = 0.008439
→ lognormal fits significantly better

power_law vs exponential
R = 176.2787
p = 0.000000
→ power_law fits significantly better


C:\Users\Parth\AppData\Roaming\Python\Python314\site-packages\powerlaw\distributions.py:808: UserWarning: Fitted parameters are very close to the edge of parameter ranges for distribution exponential; consider changing these ranges.
  warnings.warn(f'Fitted parameters are very close to the edge of parameter ranges for distribution {self.name}; consider changing these ranges.')
C:\Users\Parth\AppData\Roaming\Python\Python314\site-packages\powerlaw\distributions.py:808: UserWarning: Fitted parameters are very close to the edge of parameter ranges for distribution truncated_power_law; consider changing these ranges.
  warnings.warn(f'Fitted parameters are very close to the edge of parameter ranges for distribution {self.name}; consider changing these ranges.')



power_law vs truncated_power_law
R = -11.4771
p = 0.000002
→ truncated_power_law fits significantly better

--- Bootstrap Confidence Interval ---

Alpha = 1.451492
Bootstrap 95% CI = [1.385181, 2.145133]

Results exported successfully.


# Improved Statistical Modeling of Cooking-Action Frequencies

The original analysis assumed that the distribution of cooking-action frequencies follows a pure power-law (Zipf-like) distribution. While power-law fitting is commonly used in complex systems analysis, relying solely on visual agreement or a single fitted model can lead to misleading conclusions.

To improve the rigor of the analysis, we extended the methodology in two important ways:

---

## 1. Comparative Distribution Modeling

Instead of assuming that the data follows a power law, we statistically compared multiple candidate distributions:

- Power Law
- Lognormal
- Exponential
- Truncated Power Law

using likelihood-ratio based model comparison provided by the `powerlaw` package.

For each pair of candidate models, we computed:

- **Log-likelihood ratio (R)**  
  - Positive \(R\): first distribution fits better
  - Negative \(R\): second distribution fits better

- **p-value**
  - Small \(p\)-value indicates statistically significant preference.

### Results

| Comparison | Result |
|---|---|
| Power law vs Lognormal | Lognormal fits significantly better |
| Power law vs Exponential | Power law fits significantly better |
| Power law vs Truncated Power Law | Truncated power law fits significantly better |

---

## Interpretation

The results indicate that:

- the data is **not adequately explained by a simple exponential process**, since the power law strongly outperforms the exponential distribution.
- however, a **pure scale-free power law is also insufficient**, because both the lognormal and truncated power law provide significantly better fits.

This suggests that cooking-action organization exhibits:

- heavy-tailed behaviour,
- but with finite-size or constrained-system effects.

In real culinary systems, physical, cognitive, and procedural constraints naturally limit the unlimited scale-invariance predicted by ideal power laws. Therefore, truncated or lognormal structures may provide a more realistic generative description of cooking behaviour.

---

# Why This Improves Upon Conventional Analysis

Traditional Zipf-law analyses often:

- visually inspect log-log plots,
- fit only a power law,
- and directly report the exponent.

In contrast, our approach:

- tests competing hypotheses statistically,
- avoids assuming scale invariance a priori,
- and identifies the most plausible generative distribution.

This makes the conclusions:

- statistically stronger,
- more reproducible,
- and scientifically more defensible.

---

# 2. Bootstrap-Based Confidence Intervals

The original implementation estimated uncertainty using a normal approximation:

\[
\alpha \pm 1.96\sigma
\]

which assumes approximate Gaussian behaviour of the estimator.

To obtain a more robust estimate of uncertainty, we additionally performed bootstrap resampling:

1. repeatedly resampled the empirical data with replacement,
2. refit the model for each bootstrap sample,
3. computed the empirical distribution of the estimated exponent.

This produced a bootstrap-based 95% confidence interval:

\[
\alpha = 1.451
\]
\[
95\% \text{ CI } = [1.385,\ 2.145]
\]

---

# Significance of Bootstrap Estimation

Bootstrap estimation is advantageous because:

- it does not strongly rely on asymptotic assumptions,
- it is more robust for heavy-tailed data,
- and it better captures uncertainty in empirical complex systems.

Thus, the resulting confidence intervals are statistically more reliable than standard analytical approximations.

---

# Overall Conclusion

The extended analysis demonstrates that culinary action-frequency distributions exhibit strong heavy-tailed organization, but are better described by constrained heavy-tailed models such as truncated power laws or lognormal distributions rather than an unconstrained pure power law.

This provides a more nuanced and statistically rigorous understanding of universal statistical patterns governing culinary design.